# nimm_scd 算法使用教程

本 notebook 面向算法使用者，说明 `nimm_scd` 的目录结构、配置方式、时间拆分插件和 SCD 融合插件的调用流程。

`nimm_scd` 对外公开两个插件类：

- `QpfSplitPlugin`：时间拆分插件，把不同来源的降水产品拆成统一的 10 分钟网格文件。
- `ScdFusionPlugin`：SCD 融合插件，把拆分后的两路产品按配置权重进行融合。

真实业务运行需要准备 NetCDF/MICAPS 数据和配置路径。本教程默认只执行轻量检查与合成数据示例，不自动写业务结果。

## 0. 打开和运行环境

建议使用 `pytorch` 环境作为 notebook kernel。

如果在 PyCharm Community 里只能预览 notebook，可以在命令行运行：

```powershell
conda activate pytorch
cd D:/nimm-file/cli_code/nimm_scd/nbs
jupyter lab
```

然后在浏览器中打开 `scd_demo.ipynb`。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "nbs":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"D:/nimm-file/cli_code/nimm_scd")

WORKSPACE_ROOT = PROJECT_ROOT.parent
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

print("算法目录:", PROJECT_ROOT)
print("工作区目录:", WORKSPACE_ROOT)

## 1. 目录结构

常用目录如下：

| 目录 | 说明 |
| --- | --- |
| `src/` | 算法核心代码和公开插件类 |
| `resource/` | 默认配置文件 |
| `cli/` | 命令行入口 |
| `docs/` | 中文说明文档 |
| `test/` | 单元测试 |
| `nbs/` | notebook 教程 |

真实业务运行前，通常只需要重点修改 `resource/split_config.ini` 和 `resource/scd_pair_fusion_config.ini`。

In [ ]:
for path in [
    PROJECT_ROOT / "resource" / "split_config.ini",
    PROJECT_ROOT / "resource" / "scd_pair_fusion_config.ini",
    PROJECT_ROOT / "src" / "qpf_split_plugin.py",
    PROJECT_ROOT / "src" / "scd_fusion_plugin.py",
]:
    print(path)

## 2. 公开插件

插件类统一从 `nimm_scd.src` 导入：

```python
from nimm_scd.src import QpfSplitPlugin, ScdFusionPlugin
```

两个插件默认读取 `resource/` 下的配置文件；也可以通过 `config_path` 传入自定义配置。

In [ ]:
from nimm_scd.src import QpfSplitPlugin, ScdFusionPlugin

split_plugin = QpfSplitPlugin()
fusion_plugin = ScdFusionPlugin()

print("时间拆分配置:", split_plugin.config_path)
print("SCD 融合配置:", fusion_plugin.config_path)

## 3. 时间拆分插件 QpfSplitPlugin

时间拆分插件负责把输入产品拆成统一的 10 分钟文件，供后续 SCD 融合使用。

默认配置文件：`resource/split_config.ini`

重点配置项：

| 配置段 | 关键字段 | 说明 |
| --- | --- | --- |
| `[runtime]` | `allowed_realtime_minutes` | 实时模式允许处理的起报分钟，例如 `0,30` |
| `[source_unet_qpf]` | `base_dir` | unet_qpf 原始数据目录 |
| `[source_mait_st]` | `base_dir` | mait_st 原始数据目录 |
| `[grid]` | `time_step_minutes` | 拆分后的时间步长，当前为 10 分钟 |
| `[grid]` | `mait_extra_minutes` | mait_st 额外拆分时长，用于尾部补齐 |
| `[output]` | `base_dir` | 拆分结果输出目录 |

运行模式：

- `split_plugin.process()`：实时模式，按当前时间和 `allowed_realtime_minutes` 自动选择批次。
- `split_plugin.process(history_range=[START, END])`：历史模式，处理指定 UTC 时间范围。

时间格式固定为 UTC 的 `YYYYMMDDHHMM`。

In [ ]:
from configparser import ConfigParser

split_cfg = ConfigParser()
split_cfg.read(split_plugin.config_path, encoding="utf-8")

print("unet_qpf 输入目录:", split_cfg.get("source_unet_qpf", "base_dir", fallback=""))
print("mait_st 输入目录:", split_cfg.get("source_mait_st", "base_dir", fallback=""))
print("拆分输出目录:", split_cfg.get("output", "base_dir", fallback=""))
print("时间步长:", split_cfg.get("grid", "time_step_minutes", fallback=""), "分钟")

### 时间拆分插件调用模板

确认输入路径和输出路径配置正确后，再取消下面代码注释执行。

In [ ]:
# 实时模式
# split_plugin.process()

# 历史模式：处理 UTC 时间范围 START 到 END
# split_plugin.process(history_range=["202608080810", "202608080840"])

## 4. SCD 融合插件 ScdFusionPlugin

SCD 融合插件读取拆分后的两路 10 分钟文件，按时效配对并输出融合结果。

默认配置文件：`resource/scd_pair_fusion_config.ini`

重点配置项：

| 配置段 | 关键字段 | 说明 |
| --- | --- | --- |
| `[run]` | `dry_run` | `true` 只检查配对关系，不写结果；`false` 正常输出 |
| `[run]` | `history_step_minutes` | 历史模式遍历步长 |
| `[paths]` | `source1_dir` | 第一路拆分结果目录，通常为 unet_qpf |
| `[paths]` | `source2_dir` | 第二路拆分结果目录，通常为 mait_st |
| `[paths]` | `output_dir` | SCD 融合输出目录 |
| `[fusion]` | `blending_type` | 融合方式，当前支持普通/显著性融合逻辑 |
| `[fusion]` | `keyframe_weights` | 按时效设置 source1 权重，source2 权重为 `1 - source1` |

输出文件结构通常为：

```text
output_dir/YYYYMMDD/YYYYMMDDHHMM/YYYYMMDDHHMM.LLL.nc
```

其中 `LLL` 是预报时效分钟。

In [ ]:
fusion_cfg = ConfigParser()
fusion_cfg.read(fusion_plugin.config_path, encoding="utf-8")

print("dry_run:", fusion_cfg.get("run", "dry_run", fallback=""))
print("source1_dir:", fusion_cfg.get("paths", "source1_dir", fallback=""))
print("source2_dir:", fusion_cfg.get("paths", "source2_dir", fallback=""))
print("output_dir:", fusion_cfg.get("paths", "output_dir", fallback=""))
print("keyframe_weights:", fusion_cfg.get("fusion", "keyframe_weights", fallback=""))

## 5. 融合权重的轻量示例

下面用合成数组演示逐帧权重效果，不依赖真实业务数据。

`weights_now_list=[1.0, 0.5, 0.0]` 表示：

- 第 1 帧完全使用 source1/nowcast。
- 第 2 帧两路各 50%。
- 第 3 帧完全使用 source2/model。

In [ ]:
import numpy as np

from nimm_scd.src.linear_blending import linear_blending_forecast

source1 = np.ones((3, 4, 4), dtype=np.float32) * 10.0
source2 = np.ones((3, 4, 4), dtype=np.float32) * 2.0

blended = linear_blending_forecast(
    precomputed_nowcast=source1,
    precip_model=source2,
    weights_now_list=[1.0, 0.5, 0.0],
)

print("三帧融合后的样例值:", blended[:, 0, 0])
np.testing.assert_allclose(blended[:, 0, 0], np.array([10.0, 6.0, 2.0], dtype=np.float32))

### SCD 融合插件调用模板

确认 `source1_dir`、`source2_dir`、`output_dir` 都正确后，再取消下面代码注释执行。

建议第一次运行时先把配置里的 `dry_run = true`，只检查文件配对关系；确认无误后再改成 `false` 输出融合结果。

In [ ]:
# 实时模式
# fusion_plugin.process()

# 历史模式：处理 UTC 时间范围 START 到 END
# fusion_plugin.process(history_range=["202608080810", "202608080840"])

## 6. 命令行等价调用

如果不使用 notebook，也可以用 CLI 调用。

时间拆分：

```powershell
cd D:/nimm-file/cli_code
python -m nimm_scd.cli.split_qpf
python -m nimm_scd.cli.split_qpf 202608080810 202608080840
```

SCD 融合：

```powershell
cd D:/nimm-file/cli_code
python -m nimm_scd.cli.run_scd_fusion
python -m nimm_scd.cli.run_scd_fusion 202608080810 202608080840
```

## 7. 使用前检查清单

运行真实业务前建议逐项确认：

- `split_config.ini` 中两路原始数据目录存在。
- `split_config.ini` 中输出目录有写入权限。
- `scd_pair_fusion_config.ini` 中 `source1_dir`、`source2_dir` 指向拆分结果目录。
- `keyframe_weights` 覆盖需要融合的时效范围。
- 首次运行融合时建议 `dry_run = true`，确认配对关系后再输出结果。
- 历史模式时间使用 UTC，格式为 `YYYYMMDDHHMM`。